# Huấn luyện thử nghiệm (Overfit) trên 100 mẫu dữ liệu

Notebook này thực hiện huấn luyện mô hình Transformer Seq2Seq từ đầu trên một tập con gồm **100 mẫu ngẫu nhiên** lấy từ tập huấn luyện chính thức.

## Mục tiêu:
- Kiểm tra tính đúng đắn của toàn bộ luồng xử lý: Causal Mask, Padding Mask, Target Shift, Labels, ignore_index.
- Kiểm chứng mô hình có thể giảm loss mượt mà và ghi nhớ (overfit) được dữ liệu hay không trước khi huấn luyện trên toàn bộ bộ dữ liệu lớn.

In [ ]:
# Cài đặt các thư viện cần thiết nếu chạy trên Kaggle
!pip install -q pyyaml sentencepiece torch tqdm pandas

In [9]:
import os
import sys
import json
import random
import yaml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
import sentencepiece as spm

# Thêm thư mục gốc vào hệ thống để có thể import các file src
# Hỗ trợ cả chạy cục bộ (local) và chạy trên Kaggle
if os.path.exists("../src"):
    sys.path.append(os.path.abspath(".."))
    print("Đã liên kết thành công với thư mục dự án cục bộ.")
elif os.path.exists("./vietnamese_transformer_summarization"):
    sys.path.append(os.path.abspath("./vietnamese_transformer_summarization"))
    print("Đã liên kết với thư mục dự án trên Kaggle.")
else:
    print("Lưu ý: Hãy chắc chắn thư mục chứa src nằm trong sys.path")

from src.dataset import SummarizationDataset, SummarizationCollateFn
from src.model.transformer import TransformerSeq2Seq

Đã liên kết với thư mục dự án trên Kaggle.


In [10]:
# 1. Đọc file cấu hình YAML
config_path = "configs/transformer_summarization.yaml"
if not os.path.exists(config_path):
    config_path = "../configs/transformer_summarization.yaml"
if not os.path.exists(config_path):
    # Đường dẫn fallback khi chạy trên Kaggle
    config_path = "./vietnamese_transformer_summarization/configs/transformer_summarization.yaml"
    config_path = "./vietnamese_transformer_summarization/configs/transformer_summarization.yaml"

print(f"Đang tải cấu hình từ: {config_path}")
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

m_cfg = config["model"]
t_cfg = config["training"]
p_cfg = config["paths"]

# Định nghĩa thiết bị huấn luyện
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị sử dụng: {device}")

Đang tải cấu hình từ: ./vietnamese_transformer_summarization/configs/transformer_summarization.yaml
Thiết bị sử dụng: cuda


## Trích xuất 100 mẫu dữ liệu huấn luyện ngẫu nhiên

In [11]:
# Đường dẫn file train
train_jsonl_path = "../" + p_cfg["tokenized_train_path"]
if not os.path.exists(train_jsonl_path):
    train_jsonl_path = "./vietnamese_transformer_summarization/" + p_cfg["tokenized_train_path"]

print(f"Đang tải dữ liệu từ: {train_jsonl_path}")
full_dataset = SummarizationDataset(train_jsonl_path)
print(f"Tổng số mẫu trong tập huấn luyện gốc: {len(full_dataset):,}")

# Lấy ngẫu nhiên 100 mẫu với seed cố định để tái lập kết quả
random.seed(config["project"]["seed"])
indices = random.sample(range(len(full_dataset)), 100)
overfit_dataset = Subset(full_dataset, indices)
print(f"Đã trích xuất thành công {len(overfit_dataset)} mẫu dữ liệu thử nghiệm.")

# Cấu hình DataLoader
collate_fn = SummarizationCollateFn(pad_id=m_cfg["pad_id"])
train_loader = DataLoader(
    overfit_dataset, 
    batch_size=4,  # batch_size_per_device = 4
    shuffle=True, 
    collate_fn=collate_fn
)

Đang tải dữ liệu từ: ./vietnamese_transformer_summarization/data/bin/train_token_id.jsonl
Tổng số mẫu trong tập huấn luyện gốc: 193,841
Đã trích xuất thành công 100 mẫu dữ liệu thử nghiệm.


## Khởi tạo mô hình và Khởi tạo trọng số (Weight Initialization)

Việc khởi tạo trọng số ngẫu nhiên chuẩn phân phối (ví dụ $\sigma = 0.02$) là rất quan trọng để mô hình Transformer bắt đầu học với giá trị Loss hợp lý (khoảng 9.6 đối với bộ từ vựng 16000), tránh hiện tượng giá trị logits ban đầu bị quá lớn dẫn đến tràn số.

In [12]:
print("Đang khởi tạo mô hình TransformerSeq2Seq...")
model = TransformerSeq2Seq(
    vocab_size=m_cfg["vocab_size"],
    d_model=m_cfg["d_model"],
    num_encoder_layers=m_cfg["num_encoder_layers"],
    num_decoder_layers=m_cfg["num_decoder_layers"],
    num_heads=m_cfg["num_heads"],
    d_ff=m_cfg["d_ff"],
    max_source_length=m_cfg["max_source_length"],
    max_target_length=m_cfg["max_target_length"],
    pad_id=m_cfg["pad_id"],
    dropout=0.0,  # regularization.dropout: 0.0
    activation=m_cfg["activation"],
    share_encoder_decoder_embeddings=True,
    tie_embeddings=m_cfg.get("tie_embeddings", True)
)

# Định nghĩa hàm khởi tạo trọng số chuẩn cho Transformer
def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, nn.LayerNorm):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)

model.apply(init_weights)
model = model.to(device)
print("Khởi tạo trọng số thành công!")

Đang khởi tạo mô hình TransformerSeq2Seq...
Khởi tạo trọng số thành công!


## Cấu hình Optimizer và Loss

In [13]:
# Optimizer (AdamW: lr = 0.0005, weight_decay = 0.0)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0005,  # optimizer.learning_rate: 0.0005
    weight_decay=0.0,  # optimizer.weight_decay: 0.0
    betas=(config["optimizer"]["beta1"], config["optimizer"]["beta2"]),
    eps=config["optimizer"]["eps"]
)

# Hàm Loss bỏ qua pad_id (label_smoothing = 0.0)
criterion = nn.CrossEntropyLoss(ignore_index=t_cfg["ignore_index"], label_smoothing=0.0)

## Quá trình Huấn luyện Overfit (50 Epochs)

In [14]:
epochs = 150  # training.epochs: 150
model.train()

# Nạp tokenizer cho validation
tokenizer_path = p_cfg["tokenizer_model"]
if not os.path.exists(tokenizer_path):
    tokenizer_path = "../" + p_cfg["tokenizer_model"]
if not os.path.exists(tokenizer_path):
    tokenizer_path = "./vietnamese_transformer_summarization/" + p_cfg["tokenizer_model"]
sp = spm.SentencePieceProcessor(model_file=tokenizer_path)
verify_loader = DataLoader(overfit_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

print("Bắt đầu huấn luyện overfit trên 100 mẫu...")
for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        # Đưa dữ liệu lên thiết bị phù hợp
        src_ids = batch["source_ids"].to(device)
        src_mask = batch["source_padding_mask"].to(device)
        tgt_ids = batch["decoder_input_ids"].to(device)
        tgt_mask = batch["target_padding_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Reset gradients
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(
            src_tokens=src_ids, 
            tgt_tokens=tgt_ids, 
            src_pad_mask=src_mask, 
            tgt_pad_mask=tgt_mask
        )
        
        # Biến đổi chiều để tính Loss
        logits_flat = logits.view(-1, logits.size(-1))
        labels_flat = labels.view(-1)
        
        loss = criterion(logits_flat, labels_flat)
        
        # Backward pass và cập nhật trọng số
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * src_ids.size(0)
        
    epoch_loss = epoch_loss / len(overfit_dataset)
    
    if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
        print(f"Epoch {epoch:3d}/{epochs:3d} | Average Loss: {epoch_loss:.4f}")
        
    # validation.generate_every_epochs: 10
    if epoch % 10 == 0 or epoch == epochs:
        model.eval()
        print(f"--- KẾT QUẢ GIẢI MÃ THỬ NGHIỆM TẠI EPOCH {epoch} ---")
        with torch.no_grad():
            for idx, batch in enumerate(verify_loader):
                if idx >= 2:  # Sinh thử 2 mẫu
                    break
                src_ids = batch["source_ids"].to(device)
                src_mask = batch["source_padding_mask"].to(device)
                labels = batch["labels"].squeeze(0).tolist()
                
                enc_out = model.encode(src_ids, src_pad_mask=src_mask)
                generated = [config["tokenizer"]["bos_id"]]
                
                for _ in range(m_cfg["max_target_length"]):
                    tgt_tensor = torch.tensor([generated], dtype=torch.long, device=device)
                    logits = model.decode(tgt_tokens=tgt_tensor, enc_out=enc_out, src_pad_mask=src_mask)
                    next_token = logits[0, -1].argmax().item()
                    generated.append(next_token)
                    if next_token == config["tokenizer"]["eos_id"]:
                        break
                        
                src_text = sp.decode([t for t in src_ids.squeeze(0).tolist() if t not in [0, 3]])
                ref_text = sp.decode([t for t in labels if t not in [0, 3]])
                gen_text = sp.decode([t for t in generated if t not in [0, 2, 3]])
                
                print(f"  Mẫu {idx + 1}:")
                print(f"    - Article:   {src_text[:120]}...")
                print(f"    - Reference: {ref_text}")
                print(f"    - Generated: {gen_text}")
        print("-" * 50)

Bắt đầu huấn luyện overfit trên 100 mẫu...
Epoch   1/150 | Average Loss: 8.3497
Epoch  10/150 | Average Loss: 3.0692
--- KẾT QUẢ GIẢI MÃ THỬ NGHIỆM TẠI EPOCH 10 ---
  Mẫu 1:
    - Article:   Trong khi thực hiện chuyến bay lần thứ 11 trên hành tinh đỏ vào tuần trước, chiếc trực thăng nhỏ đã bắt gặp hình ảnh "tà...
    - Reference: Trực thăng sao Hỏa Ingenuity của NASA phát hiện "tàu mẹ" Perseverance từ trên cao trong một khung cảnh hoành tráng.
    - Generated: Chiều Chiều Chiều Chiều 10.5, Công an tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh tỉnh, các tỉnh, người đã tới tới tới những những những những những những những những những những những những những những những những những những những những những những những những những những những những những những những những năm.
  Mẫu 2:
    - Article:   Tính từ ngày 05.06.2021 đến nay, Vĩnh Phúc có 15 ngày liên tiếp chưa phát hiện trường hợp mắc/nghi ngờ mắc COVID-19 trên...
    - Reference: Ban Chỉ đạo phòng, ch

## Giải mã kiểm thử (Inference Verification)

Thử nghiệm chạy sinh câu tóm tắt (Greedy Decoding) trên 2 mẫu trong tập 100 câu này để xem mô hình đã thuộc lòng (memorize) chính xác câu tóm tắt mục tiêu hay chưa.

In [30]:
# Load tokenizer
tokenizer_path = p_cfg["tokenizer_model"]
if not os.path.exists(tokenizer_path):
    tokenizer_path = "../" + p_cfg["tokenizer_model"]
if not os.path.exists(tokenizer_path):
    tokenizer_path = "./vietnamese_transformer_summarization/" + p_cfg["tokenizer_model"]
    
sp = spm.SentencePieceProcessor(model_file=tokenizer_path)

model.eval()

# Lấy ngẫu nhiên 2 mẫu để sinh thử
verify_loader = DataLoader(overfit_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

print("\n--- KẾT QUẢ GIẢI MÃ THỬ NGHIỆM (GREEDY GENERATION) ---")
with torch.no_grad():
    for idx, batch in enumerate(verify_loader):
        if idx >= 2:
            break
            
        src_ids = batch["source_ids"].to(device)
        src_mask = batch["source_padding_mask"].to(device)
        labels = batch["labels"].squeeze(0).tolist()
        
        # 1. Mã hóa văn bản gốc bằng encoder
        enc_out = model.encode(src_ids, src_pad_mask=src_mask)
        
        # 2. Khởi tạo đầu vào decoder với token <bos> (ID 2)
        generated = [config["tokenizer"]["bos_id"]]
        
        # 3. Sinh tự hồi quy (autoregressive greedy decoding)
        for _ in range(m_cfg["max_target_length"]):
            tgt_tensor = torch.tensor([generated], dtype=torch.long, device=device)
            logits = model.decode(tgt_tokens=tgt_tensor, enc_out=enc_out, src_pad_mask=src_mask)
            
            # Lấy token có xác suất cao nhất tại bước cuối cùng
            next_token = logits[0, -1].argmax().item()
            generated.append(next_token)
            
            # Dừng nếu gặp token <eos> (ID 3)
            if next_token == config["tokenizer"]["eos_id"]:
                break
                
        # Giải mã các mã token ID về chữ viết tiếng Việt
        src_text = sp.decode([t for t in src_ids.squeeze(0).tolist() if t not in [0, 3]])
        ref_text = sp.decode([t for t in labels if t not in [0, 3]])
        gen_text = sp.decode([t for t in generated if t not in [0, 2, 3]])
        
        print(f"\nMẫu thử số {idx + 1}:")
        print(f"  - Bài viết gốc (rút gọn): {src_text[:200]}...")
        print(f"  - Tóm tắt gốc (Reference): {ref_text}")
        print(f"  - Mô hình sinh (Generated): {gen_text}")
        print("-" * 50)


--- KẾT QUẢ GIẢI MÃ THỬ NGHIỆM (GREEDY GENERATION) ---

Mẫu thử số 1:
  - Bài viết gốc (rút gọn): Ngày 10.8, Phòng Cảnh sát hình sự Công an tỉnh Bắc Giang cho biết đang tạm giữ hình sự 7 đối tượng để điều tra, xử lý về hành vi liên quan đến đánh bạc trên mạng Internet. Trước đó, chiều 7.8, Phòng C...
  - Tóm tắt gốc (Reference): Đào Ngọc Quân, ở Bắc Giang mua bán tiền ảo để đổ vào trò chơi điện tử R88, đánh bạc trên mạng Internet với số tiền thật lên tới cả chục tỉ đồng.
  - Mô hình sinh (Generated): Đào Ngọc Quân, ở Bắc Giang mua bán tiền ảo để đổ vào trò chơi điện tử R88, đánh bạc trên mạng Internet với số tiền thật lên tới cả chục tỉ đồng.
--------------------------------------------------

Mẫu thử số 2:
  - Bài viết gốc (rút gọn): Những thay đổi hiếm thấy Với việc hàng loạt cầu thủ như Tuấn Anh, Đức Chinh, Duy Mạnh gặp chấn thương, Văn Toàn, Tiến Linh vắng mặt vì COVID-19, huấn luyện viên Park Hang-seo phải tiến hành hàng loạt ...
  - Tóm tắt gốc (Reference): Trận đấu với tuyển Au